# Battlecode experiment worker (Kaggle)

Runs the shared experiment queue (`league/experiments.json`) like `scripts\\train.ps1 experiments`
on a PC: claims an idea, tests it against the mimics, pushes the result.

**One-time setup (see docs/league.pdf, 'Kaggle'):**
1. Settings (right panel): **Internet on** (needs a phone-verified account), Accelerator **None** (CPU).
2. Add-ons -> Secrets: add `GITHUB_TOKEN` = a GitHub token with write access to the repo,
   and tick it for this notebook.
3. Change `NAME` below if you run more than one Kaggle session at once.
4. **Save Version -> Save & Run All (Commit)**: it runs up to 12 h with the browser closed.


In [ ]:
NAME = 'kaggle-1'          # this worker's name in exp-status (unique per running session)
HOURS = 11                 # Kaggle stops sessions at 12 h; leave time to hand back the last claim
REPO = 'github.com/Ahmed-Abdul-Rahim/battlecode-2026a.git'
BRANCH = 'league'
WORK = '/kaggle/working/battlecode'


In [ ]:
import os, subprocess, sys, time, threading
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')

def sh(cmd, cwd=None, check=True):
    print('$', cmd.replace(token, '***'))
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    out = (r.stdout + r.stderr).replace(token, '***')
    print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'failed: {cmd[:60]}')
    return out

# Rust (the exact toolchain comes from the repo's rust-toolchain.toml) and cambc
sh('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain none')
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
sh(f'{sys.executable} -m pip install -q "cambc>=3.0.1"')

# The repo, with push access
if not os.path.isdir(WORK):
    sh(f'git clone -q -b {BRANCH} https://x-access-token:{token}@{REPO} {WORK}')
sh('git config user.name "battlecode-kaggle" && git config user.email "kaggle@users.noreply.github.com"', cwd=WORK)
sh('rustup show', cwd=WORK)                       # installs 1.99.0 + wasm32-wasip1
sh(f'{sys.executable} -m pip install -q -e .', cwd=WORK)
sh('cambc --version', cwd=WORK)
sh(f'{sys.executable} harness/params.py --check', cwd=WORK)
print(f'cores: {os.cpu_count()}')


In [ ]:
# Run the queue. Each experiment deletes its games when done; at the time limit the
# unfinished one is handed back so another PC can take it at once.
env = dict(os.environ, LEAGUE_HOST=NAME, BCTOOLS_JOBS=str(os.cpu_count()), PYTHONUTF8='1')
p = subprocess.Popen([sys.executable, '-u', 'harness/experiments.py', 'run', '--forever', '--hours', str(HOURS)],
                     cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
timer = threading.Timer((HOURS + 0.5) * 3600, p.kill)   # hard stop before Kaggle's 12 h
timer.start()
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()
timer.cancel()
print('worker stopped, exit', p.returncode)


In [ ]:
# Hand back anything unfinished and show the queue.
subprocess.run([sys.executable, 'harness/experiments.py', 'release'], cwd=WORK, env=env)
subprocess.run([sys.executable, 'harness/experiments.py', 'status'], cwd=WORK, env=env)
